In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd, io, re
from pathlib import Path

# Correction du chemin d'accès pour pointer vers le bon dossier et fichier
chemin = Path('/content/drive/MyDrive/SANAE-ELRHARABY-BASMA-EN-NIYARI-TP2/FMI XLS.xls')
if not chemin.is_file():
    raise FileNotFoundError(f'Fichier introuvable : {chemin}')

# 1) lire le texte (essayer plusieurs encodages)
texte = None
for enc in ('utf-8-sig', 'utf-16', 'cp1252', 'latin-1'):
    try:
        texte = chemin.read_text(encoding=enc)
        break
    except UnicodeError:
        continue

lignes = [l for l in texte.splitlines() if l.strip()]

# 2) trouver la ligne d'en-tête (celle qui commence par "Date")
i_entete = next(i for i, l in enumerate(lignes) if l.strip().startswith('Date'))

# 3) garder seulement les lignes de données (commencent par "Mois jj, aaaa")
motif = re.compile(r'^\s*[A-Z][a-z]+ \d{1,2}, \d{4}\t')
donnees = [l for l in lignes[i_entete + 1:] if motif.match(l)]

df = pd.read_csv(io.StringIO('\n'.join([lignes[i_entete]] + donnees)),
                 sep='\t', thousands=',')
df.columns = [c.strip() for c in df.columns]

df = df.rename(columns={
    'Date': 'date',
    'Stand-by/Credit Tranche/Emergency': 'sba',
    'Extended Fund Facility': 'eff',
    'Flexible Credit Line and Precautionary and Liquidity Line 1/': 'fcl_pll',
    'Short-Term Liquidity Line': 'stll',
    'PRGT': 'prgt', 'RST': 'rst',
    'SAF/Trust Fund': 'saf_trust', 'Other': 'other', 'Total': 'total'})

df['date'] = pd.to_datetime(df['date'].str.strip(), format='%B %d, %Y')
df['annee'] = df['date'].dt.year

display(df.head(10))
print('Dimensions :', df.shape)

,date,sba,eff,fcl_pll,stll,prgt,rst,saf_trust,other,total,annee
0,2026-09-30,0,0,0,0,0,937500000,0,0,937500000,2026
1,2025-12-31,0,0,0,0,0,937500000,0,0,937500000,2025
2,2024-12-31,0,0,537700000,0,0,562500000,0,0,1100200000,2024
3,2023-12-31,0,0,1499800000,0,0,0,0,0,1499800000,2023
4,2022-12-31,0,0,1499800000,0,0,0,0,0,1499800000,2022
5,2021-12-31,0,0,1499800000,0,0,0,0,0,1499800000,2021
6,2020-12-31,0,0,2150800000,0,0,0,0,0,2150800000,2020
7,1996-12-31,2299500,0,0,0,0,0,0,0,2299500,1996
8,1995-12-31,34830832,0,0,0,0,0,0,0,34830832,1995
9,1994-12-31,101062662,0,0,0,0,0,0,0,101062662,1994


Dimensions : (20, 11)


In [10]:
import sqlite3
import pandas as pd

# 1. Créer une connexion de base de données SQLite en mémoire
conn = sqlite3.connect(':memory:')

# 2. Charger le DataFrame 'df' créé précédemment dans la table 'fmi_maroc'
df.to_sql('fmi_maroc', conn, index=False, if_exists='replace')

# Fonction utilitaire pour exécuter et afficher le résultat d'une requête SQL
def executer_sql(titre, requete):
    print(f"=== {titre} ===")
    resultat = pd.read_sql_query(requete, conn)
    display(resultat)
    print("\n")

# 1. Toutes les observations
executer_sql("1. Toutes les observations", "SELECT * FROM fmi_maroc;")

# 2. Certaines colonnes
executer_sql("2. Certaines colonnes", "SELECT annee, sba, eff, fcl_pll, prgt, rst, total FROM fmi_maroc;")

# 3. Limiter les résultats
executer_sql("3. Limiter les résultats", "SELECT annee, total FROM fmi_maroc LIMIT 5;")

# 4. Trier (du plus grand au plus petit encours)
executer_sql("4. Trier (du plus grand au plus petit encours)", "SELECT annee, total FROM fmi_maroc ORDER BY total DESC;")

# 5. Filtrer avec WHERE
executer_sql("5. Filtrer avec WHERE", "SELECT annee, total FROM fmi_maroc WHERE annee >= 2020 ORDER BY annee;")

# 6. Nombre total d'enregistrements
executer_sql("6. Nombre total d'enregistrements", "SELECT COUNT(*) AS nb_enregistrements FROM fmi_maroc;")

# 7. Moyenne
executer_sql("7. Moyenne", "SELECT ROUND(AVG(total)) AS moyenne_total FROM fmi_maroc;")

# 8. Minimum et maximum
executer_sql("8. Minimum et maximum", "SELECT MIN(total) AS min_total, MAX(total) AS max_total FROM fmi_maroc;")

# 9. GROUP BY (par décennie)
executer_sql("9. GROUP BY (par décennie)", """
SELECT (annee/10)*10 AS decennie,
       COUNT(*) AS nb,
       ROUND(AVG(total)) AS moyenne_total
FROM fmi_maroc
GROUP BY decennie
ORDER BY decennie;
""")

# 10. HAVING (décennies avec moyenne > 1 milliard de DTS)
executer_sql("10. HAVING (décennies avec moyenne > 1 milliard de DTS)", """
SELECT (annee/10)*10 AS decennie,
       ROUND(AVG(total)) AS moyenne_total
FROM fmi_maroc
GROUP BY decennie
HAVING AVG(total) > 1000000000
ORDER BY decennie;
""")

# 11. AND / OR (années où SBA et EFF étaient actifs ensemble)
executer_sql("11. AND / OR (années où SBA et EFF étaient actifs ensemble)", """
SELECT annee, sba, eff, total
FROM fmi_maroc
WHERE sba > 0 AND eff > 0
ORDER BY annee;
""")

# 12. BETWEEN (ou IN)
executer_sql("12. BETWEEN", """
SELECT annee, total FROM fmi_maroc
WHERE annee BETWEEN 1985 AND 1990
ORDER BY annee;
""")

executer_sql("12. IN", """
SELECT annee, total FROM fmi_maroc
WHERE annee IN (1984, 1990, 2020, 2024)
ORDER BY annee;
""")

# 13. Classement
executer_sql("13. Classement", """
SELECT annee, total,
       RANK() OVER (ORDER BY total DESC) AS rang
FROM fmi_maroc
ORDER BY rang
LIMIT 10;
""")

# 14. Sous-requête (années au-dessus de la moyenne)
executer_sql("14. Sous-requête (années au-dessus de la moyenne)", """
SELECT annee, total
FROM fmi_maroc
WHERE total > (SELECT AVG(total) FROM fmi_maroc)
ORDER BY total DESC;
""")

# 15. Synthèse : quel instrument domine chaque année ?
executer_sql("15. Synthèse : quel instrument domine chaque année ?", """
SELECT annee, total,
  CASE WHEN sba > 0     AND sba >= eff AND sba >= fcl_pll AND sba >= prgt AND sba >= rst THEN 'SBA'
       WHEN eff > 0     AND eff >= sba AND eff >= fcl_pll AND eff >= prgt AND eff >= rst THEN 'EFF'
       WHEN fcl_pll > 0 AND fcl_pll >= sba AND fcl_pll >= eff AND fcl_pll >= prgt AND fcl_pll >= rst THEN 'FCL/PLL'
       WHEN prgt > 0    AND prgt >= sba AND prgt >= eff AND prgt >= fcl_pll AND prgt >= rst THEN 'PRGT'
       WHEN rst > 0 THEN 'RST'
       ELSE 'Autre' END AS instrument_dominant
FROM fmi_maroc
ORDER BY annee DESC;
""")

=== 1. Toutes les observations ===


,date,sba,eff,fcl_pll,stll,prgt,rst,saf_trust,other,total,annee
0,2026-09-30 00:00:00,0,0,0,0,0,937500000,0,0,937500000,2026
1,2025-12-31 00:00:00,0,0,0,0,0,937500000,0,0,937500000,2025
2,2024-12-31 00:00:00,0,0,537700000,0,0,562500000,0,0,1100200000,2024
3,2023-12-31 00:00:00,0,0,1499800000,0,0,0,0,0,1499800000,2023
4,2022-12-31 00:00:00,0,0,1499800000,0,0,0,0,0,1499800000,2022
5,2021-12-31 00:00:00,0,0,1499800000,0,0,0,0,0,1499800000,2021
6,2020-12-31 00:00:00,0,0,2150800000,0,0,0,0,0,2150800000,2020
7,1996-12-31 00:00:00,2299500,0,0,0,0,0,0,0,2299500,1996
8,1995-12-31 00:00:00,34830832,0,0,0,0,0,0,0,34830832,1995
9,1994-12-31 00:00:00,101062662,0,0,0,0,0,0,0,101062662,1994




=== 2. Certaines colonnes ===


,annee,sba,eff,fcl_pll,prgt,rst,total
0,2026,0,0,0,0,937500000,937500000
1,2025,0,0,0,0,937500000,937500000
2,2024,0,0,537700000,0,562500000,1100200000
3,2023,0,0,1499800000,0,0,1499800000
4,2022,0,0,1499800000,0,0,1499800000
5,2021,0,0,1499800000,0,0,1499800000
6,2020,0,0,2150800000,0,0,2150800000
7,1996,2299500,0,0,0,0,2299500
8,1995,34830832,0,0,0,0,34830832
9,1994,101062662,0,0,0,0,101062662




=== 3. Limiter les résultats ===


...

,annee,total,rang
0,2020,2150800000,1
1,2023,1499800000,2
2,2022,1499800000,2
3,2021,1499800000,2
4,1985,1160563782,5
5,1984,1107197381,6
6,2024,1100200000,7
7,2026,937500000,8
8,2025,937500000,8
9,1986,894386478,10




=== 14. Sous-requête (années au-dessus de la moyenne) ===


,annee,total
0,2020,2150800000
1,2023,1499800000
2,2022,1499800000
3,2021,1499800000
4,1985,1160563782
5,1984,1107197381
6,2024,1100200000
7,2026,937500000
8,2025,937500000
9,1986,894386478




=== 15. Synthèse : quel instrument domine chaque année ? ===


,annee,total,instrument_dominant
0,2026,937500000,RST
1,2025,937500000,RST
2,2024,1100200000,RST
3,2023,1499800000,FCL/PLL
4,2022,1499800000,FCL/PLL
5,2021,1499800000,FCL/PLL
6,2020,2150800000,FCL/PLL
7,1996,2299500,SBA
8,1995,34830832,SBA
9,1994,101062662,SBA
